In [1]:
import json
import numpy as np
import pandas as pd
import os
from os.path import join, dirname
import pandas as pd
from copy import deepcopy
from datetime import date
from dateutil.relativedelta import relativedelta
import math
import json

In [44]:
def DeferralRPS(
        FinanceAmount: float,
        ProfitRate: float,
        PayDay : int,
        DisbursementDate : str,
        TakafulFactor : float,
        EMI : float,
        GracePeriodMonths : int,
        repaymentMethod : str = "Deferral"
        ):
    
    DisbursementYear = int(DisbursementDate[0:4])
    DisbursementMonth = int(DisbursementDate[4:6])
    DisbursementDay = int(DisbursementDate[6:8])
    
    gracePeriodEndDate = date(DisbursementYear, DisbursementMonth, DisbursementDay) + relativedelta(months=GracePeriodMonths)
    
    if gracePeriodEndDate.day >= PayDay:
        FirstEMIYear = (gracePeriodEndDate + relativedelta(months=1)).year
        FirstEMIMonth = (gracePeriodEndDate + relativedelta(months=1)).month
    else:
        FirstEMIYear = gracePeriodEndDate.year
        FirstEMIMonth = gracePeriodEndDate.month
    
    if repaymentMethod == "Deferral":
        rps = {
            'SNo' : [],
            'Date' : [],
            'Days' : [],
            'EMI' : [],
            "GracePeriodProfitRecovery" : [],
            "GracePeriodTakafulRecovery" : [],
            'ProfitAmount' : [],
            'TakafulAmount' : [],
            'PrincipalAmount' : [],
            'OutstandingPrincipal' : []
           }

        distributionMonths = 1

        if GracePeriodMonths == 0:
            raise ValueError("Grace period cannot be 0 for Deferral method.")
        
        # Create the first row of the RPS to show the disbursement of the loan
        rps['SNo'].append(0)
        rps['Date'].append(date(DisbursementYear, DisbursementMonth, DisbursementDay))
        rps['Days'].append(0)
        rps['EMI'].append(0)
        rps['ProfitAmount'].append(0)
        rps['TakafulAmount'].append(0)
        rps['GracePeriodTakafulRecovery'].append(0)
        rps['GracePeriodProfitRecovery'].append(0)
        rps['PrincipalAmount'].append(0)
        rps['OutstandingPrincipal'].append(FinanceAmount)



        GracePeriodYear = gracePeriodEndDate.year
        GracePeriodMonth = gracePeriodEndDate.month
        GracePeriodDay = DisbursementDay

        # Create the second row of the RPS to show the end of the grace period
        rps['SNo'].append(1)
        rps['Date'].append(date(GracePeriodYear, GracePeriodMonth, GracePeriodDay))  
        rps['Days'].append((rps['Date'][-1] - rps['Date'][-2]).days)
        rps['EMI'].append(0)
        rps['ProfitAmount'].append(0)
        rps['TakafulAmount'].append(0)
        rps['GracePeriodTakafulRecovery'].append(0)
        rps['GracePeriodProfitRecovery'].append(0)
        rps['PrincipalAmount'].append(0)
        rps['OutstandingPrincipal'].append(rps['OutstandingPrincipal'][-1])


        # GracePeriodTakaful = rps['OutstandingPrincipal'][-1] * TakafulFactor * rps['Days'][-1] / 30
        GracePeriodTakaful = round(rps['OutstandingPrincipal'][-1] * TakafulFactor * GracePeriodMonths,3)
        GracePeriodProfit = round(rps['OutstandingPrincipal'][-1] * ProfitRate / 360 * rps['Days'][-1],3)


        # Calculate how many months will it take to pay off the Deferred Profit and Takaful amounts.
        while True:
            if distributionMonths > 360:
                raise ValueError("Installment too low to cater to deferrment with the given Profit Rate, or requires more than 360 installments.")


            rps_copy = deepcopy(rps)
            for i in range(0,distributionMonths):
                year = FirstEMIYear + ((FirstEMIMonth + (i-1))//12)
                month = ((FirstEMIMonth + (i-1)) % 12) + 1
                day = PayDay
                rps_copy['Date'].append(date(year, month, day))  
                rps_copy['Days'].append((rps_copy['Date'][-1] - rps_copy['Date'][-2]).days)
                rps_copy['ProfitAmount'].append(round(rps_copy['OutstandingPrincipal'][-1] * ProfitRate / 360 * rps_copy['Days'][-1],3))
                # rps_copy['TakafulAmount'].append(rps_copy['OutstandingPrincipal'][-1] * TakafulFactor * rps_copy['Days'][-1] / 30)
                rps_copy['TakafulAmount'].append(round(rps_copy['OutstandingPrincipal'][-1] * TakafulFactor,3))
                rps_copy['GracePeriodTakafulRecovery'].append(round(GracePeriodTakaful / distributionMonths,3))
                rps_copy['GracePeriodProfitRecovery'].append(round(GracePeriodProfit / distributionMonths,3))
                rps_copy['EMI'].append(EMI)
                rps_copy['PrincipalAmount'].append(round(rps_copy['EMI'][-1] - rps_copy['ProfitAmount'][-1] - rps_copy['TakafulAmount'][-1] - rps_copy['GracePeriodTakafulRecovery'][-1] - rps_copy['GracePeriodProfitRecovery'][-1],3))
                rps_copy['OutstandingPrincipal'].append(round(rps_copy['OutstandingPrincipal'][-1] - rps_copy['PrincipalAmount'][-1],3))

            if rps_copy['OutstandingPrincipal'][-1] < 0:
                EMI = rps_copy['EMI'][-1] + rps_copy['OutstandingPrincipal'][-1]
                PrincipalAmount = rps_copy['PrincipalAmount'][-1] + rps_copy['OutstandingPrincipal'][-1]
                rps_copy['EMI'][-1] = EMI
                rps_copy['PrincipalAmount'][-1] = PrincipalAmount
                rps_copy['OutstandingPrincipal'][-1] = 0
                break


            if sum(1 for i in rps_copy['PrincipalAmount'] if i < 0)== 0:
                break
            else:
                distributionMonths += 1

        # In case the loan is fully paid off during the distribution period, then we just copy the RPS without adding more rows to it. 
        if rps_copy['OutstandingPrincipal'][-1] > 0:
            
            # Add the rows for the period during which we are recovering the deferred profit and takaful amounts.
            for i in range (0, distributionMonths):
                rps['SNo'].append(2+i) 
                year = FirstEMIYear + ((FirstEMIMonth + (i-1))//12)
                month = ((FirstEMIMonth + (i-1)) % 12) + 1
                day = PayDay
                rps['Date'].append(date(year, month, day))  
                rps['Days'].append((rps['Date'][-1] - rps['Date'][-2]).days)
                rps['ProfitAmount'].append(round(rps['OutstandingPrincipal'][-1] * ProfitRate / 360 * rps['Days'][-1],3))
                # rps['TakafulAmount'].append(rps['OutstandingPrincipal'][-1] * TakafulFactor * rps['Days'][-1] / 30)
                rps['TakafulAmount'].append(round(rps['OutstandingPrincipal'][-1] * TakafulFactor,3))
                rps['GracePeriodTakafulRecovery'].append(round(GracePeriodTakaful / distributionMonths,3))
                rps['GracePeriodProfitRecovery'].append(round(GracePeriodProfit / distributionMonths,3))
                rps['EMI'].append(EMI)
                rps['PrincipalAmount'].append(round(max(0,rps['EMI'][-1] - rps['ProfitAmount'][-1] - rps['TakafulAmount'][-1] - rps['GracePeriodTakafulRecovery'][-1] - rps['GracePeriodProfitRecovery'][-1]),3))
                rps['OutstandingPrincipal'].append(round(rps['OutstandingPrincipal'][-1] - rps['PrincipalAmount'][-1],3))
                    

            i = distributionMonths

            while rps['OutstandingPrincipal'][-1] > 0:
                rps['SNo'].append(rps['SNo'][-1] + 1)
                year = FirstEMIYear + ((FirstEMIMonth + (i-1))//12)
                month = ((FirstEMIMonth + (i-1)) % 12) + 1
                day = PayDay
                rps['Date'].append(date(year, month, day))  
                
                rps['Days'].append((rps['Date'][-1] - rps['Date'][-2]).days)
                
                rps['EMI'].append(EMI)
                rps['ProfitAmount'].append(round(rps['OutstandingPrincipal'][-1] * ProfitRate / 360 * rps['Days'][-1],3))
                # rps['TakafulAmount'].append(rps['OutstandingPrincipal'][-1] * TakafulFactor * rps['Days'][-1] / 30)
                rps['TakafulAmount'].append(round(rps['OutstandingPrincipal'][-1] * TakafulFactor,3))
                rps['GracePeriodTakafulRecovery'].append(0)
                rps['GracePeriodProfitRecovery'].append(0)
                rps['PrincipalAmount'].append(round(max(0,rps['EMI'][-1] - rps['ProfitAmount'][-1] - rps['TakafulAmount'][-1] - rps['GracePeriodTakafulRecovery'][-1] - rps['GracePeriodProfitRecovery'][-1]),3))
                rps['OutstandingPrincipal'].append(round(rps['OutstandingPrincipal'][-1] - rps['PrincipalAmount'][-1],3))

                i += 1

            if rps['OutstandingPrincipal'][-1] < 0:
                EMI = rps['EMI'][-1] + rps['OutstandingPrincipal'][-1]
                PrincipalAmount = rps['PrincipalAmount'][-1] + rps['OutstandingPrincipal'][-1]
                rps['EMI'][-1] = EMI
                rps['PrincipalAmount'][-1] = PrincipalAmount
                rps['OutstandingPrincipal'][-1] = 0
                
        else:
            rps = rps_copy


    else:
        raise AttributeError("Please input the correct repaymentMethod. (Deferral)")
    
    df = pd.DataFrame(rps)

    return df

In [45]:
DeferralRPS(
        FinanceAmount = 5000,
        ProfitRate = 0.05,
        PayDay = 1,
        DisbursementDate = "20260415",
        TakafulFactor  = 0.055,
        EMI = 1000,
        GracePeriodMonths = 3,
        repaymentMethod = "Deferral"
        )

,SNo,Date,Days,EMI,GracePeriodProfitRecovery,GracePeriodTakafulRecovery,ProfitAmount,TakafulAmount,PrincipalAmount,OutstandingPrincipal
0,0,2026-04-15,0,0.000,0.000,0.0,0.000,0.000,0.000,5000.000
1,1,2026-07-15,91,0.000,0.000,0.0,0.000,0.000,0.000,5000.000
2,2,2026-08-01,17,1000.000,31.597,412.5,11.806,275.000,269.097,4730.903
3,3,2026-09-01,31,1000.000,31.597,412.5,20.369,260.200,275.334,4455.569
4,4,2026-10-01,30,1000.000,0.000,0.0,18.565,245.056,736.379,3719.190
5,5,2026-11-01,31,1000.000,0.000,0.0,16.013,204.555,779.432,2939.758
6,6,2026-12-01,30,1000.000,0.000,0.0,12.249,161.687,826.064,2113.694
7,7,2027-01-01,31,1000.000,0.000,0.0,9.101,116.253,874.646,1239.048
8,8,2027-02-01,31,1000.000,0.000,0.0,5.335,68.148,926.517,312.531
9,9,2027-03-01,28,330.935,0.000,0.0,1.215,17.189,312.531,0.000


In [57]:
def GoalSeekFA(EMI, ProfitRate, TakafulFactor, Tenor, PayDay, 
               DisbursementDate, GracePeriodMonths, tolerance=0.001):
    
    low, high = 0.0, EMI * Tenor  # upper bound: if 0% rate, FA = EMI × N
    
    for _ in range(100):  # binary search, converges fast
        mid = (low + high) / 2
        
        try:
            result = DeferralRPS(
                FinanceAmount=mid,
                ProfitRate=ProfitRate,
                PayDay=PayDay,
                DisbursementDate=DisbursementDate,
                TakafulFactor=TakafulFactor,
                EMI=EMI,
                GracePeriodMonths=GracePeriodMonths
            )
        except ValueError:
            high = mid  # if installment too low, reduce FA
            continue

        actual_tenor = len(result) - 2   # subtract disbursement + grace rows
        
        if abs(actual_tenor - Tenor) <= tolerance:
            return mid
        elif actual_tenor > Tenor:  # FA too high, loan takes too long
            high = mid
        else:                       # FA too low, loan pays off too fast
            low = mid
    
    # After 100 iterations, if low and high haven't converged meaningfully,
    # no valid solution exists for these inputs
    if high - low < tolerance:
        raise ValueError(
            f"No valid FinanceAmount exists for EMI={EMI}, Tenor={Tenor}, "
            f"GracePeriodMonths={GracePeriodMonths}. EMI is too low to service "
            f"the profit and takaful charges."
        )

    return mid

In [58]:
def SeekPrincipal(
        Tenor: float,
        ProfitRate: float,
        PayDay : int,
        DisbursementDate : str,
        TakafulFactor : float,
        EMI : float,
        GracePeriodMonths : int,
        repaymentMethod : str = "Deferral"
        ):

    if TakafulFactor == 0 and GracePeriodMonths == 0:
        FinanceAmount = EMI * (1 - (1 + ProfitRate/12) ** (-Tenor)) / (ProfitRate/12)
    elif TakafulFactor != 0 and GracePeriodMonths == 0:
        EffectiveRate = (ProfitRate/12) + TakafulFactor
        FinanceAmount = EMI * (1 - (1 + EffectiveRate)**(-Tenor)) / (EffectiveRate)
    elif GracePeriodMonths != 0:
        FinanceAmount = GoalSeekFA(EMI, ProfitRate, TakafulFactor, Tenor, PayDay, DisbursementDate, GracePeriodMonths)
    else:
        raise ValueError("Invalid combination of TakafulFactor and GracePeriodMonths.")
        
    
    DisbursementYear = int(DisbursementDate[0:4])
    DisbursementMonth = int(DisbursementDate[4:6])
    DisbursementDay = int(DisbursementDate[6:8])
    
    gracePeriodEndDate = date(DisbursementYear, DisbursementMonth, DisbursementDay) + relativedelta(months=GracePeriodMonths)
    
    if gracePeriodEndDate.day >= PayDay:
        FirstEMIYear = (gracePeriodEndDate + relativedelta(months=1)).year
        FirstEMIMonth = (gracePeriodEndDate + relativedelta(months=1)).month
    else:
        FirstEMIYear = gracePeriodEndDate.year
        FirstEMIMonth = gracePeriodEndDate.month
    
    if repaymentMethod == "Deferral":
        rps = {
            'SNo' : [],
            'Date' : [],
            'Days' : [],
            'EMI' : [],
            "GracePeriodProfitRecovery" : [],
            "GracePeriodTakafulRecovery" : [],
            'ProfitAmount' : [],
            'TakafulAmount' : [],
            'PrincipalAmount' : [],
            'OutstandingPrincipal' : []
           }

        distributionMonths = 1

        if GracePeriodMonths == 0:
            raise ValueError("Grace period cannot be 0 for Deferral method.")
        
        # Create the first row of the RPS to show the disbursement of the loan
        rps['SNo'].append(0)
        rps['Date'].append(date(DisbursementYear, DisbursementMonth, DisbursementDay))
        rps['Days'].append(0)
        rps['EMI'].append(0)
        rps['ProfitAmount'].append(0)
        rps['TakafulAmount'].append(0)
        rps['GracePeriodTakafulRecovery'].append(0)
        rps['GracePeriodProfitRecovery'].append(0)
        rps['PrincipalAmount'].append(0)
        rps['OutstandingPrincipal'].append(FinanceAmount)



        GracePeriodYear = gracePeriodEndDate.year
        GracePeriodMonth = gracePeriodEndDate.month
        GracePeriodDay = DisbursementDay

        # Create the second row of the RPS to show the end of the grace period
        rps['SNo'].append(1)
        rps['Date'].append(date(GracePeriodYear, GracePeriodMonth, GracePeriodDay))  
        rps['Days'].append((rps['Date'][-1] - rps['Date'][-2]).days)
        rps['EMI'].append(0)
        rps['ProfitAmount'].append(0)
        rps['TakafulAmount'].append(0)
        rps['GracePeriodTakafulRecovery'].append(0)
        rps['GracePeriodProfitRecovery'].append(0)
        rps['PrincipalAmount'].append(0)
        rps['OutstandingPrincipal'].append(rps['OutstandingPrincipal'][-1])


        # GracePeriodTakaful = rps['OutstandingPrincipal'][-1] * TakafulFactor * rps['Days'][-1] / 30
        GracePeriodTakaful = round(rps['OutstandingPrincipal'][-1] * TakafulFactor * GracePeriodMonths,3)
        GracePeriodProfit = round(rps['OutstandingPrincipal'][-1] * ProfitRate / 360 * rps['Days'][-1],3)


        # Calculate how many months will it take to pay off the Deferred Profit and Takaful amounts.
        while True:
            if distributionMonths > 360:
                raise ValueError("Installment too low to cater to deferrment with the given Profit Rate.")


            rps_copy = deepcopy(rps)
            for i in range(0,distributionMonths):
                year = FirstEMIYear + ((FirstEMIMonth + (i-1))//12)
                month = ((FirstEMIMonth + (i-1)) % 12) + 1
                day = PayDay
                rps_copy['Date'].append(date(year, month, day))  
                rps_copy['Days'].append((rps_copy['Date'][-1] - rps_copy['Date'][-2]).days)
                rps_copy['ProfitAmount'].append(round(rps_copy['OutstandingPrincipal'][-1] * ProfitRate / 360 * rps_copy['Days'][-1],3))
                # rps_copy['TakafulAmount'].append(rps_copy['OutstandingPrincipal'][-1] * TakafulFactor * rps_copy['Days'][-1] / 30)
                rps_copy['TakafulAmount'].append(round(rps_copy['OutstandingPrincipal'][-1] * TakafulFactor,3))
                rps_copy['GracePeriodTakafulRecovery'].append(round(GracePeriodTakaful / distributionMonths,3))
                rps_copy['GracePeriodProfitRecovery'].append(round(GracePeriodProfit / distributionMonths,3))
                rps_copy['EMI'].append(EMI)
                rps_copy['PrincipalAmount'].append(round(rps_copy['EMI'][-1] - rps_copy['ProfitAmount'][-1] - rps_copy['TakafulAmount'][-1] - rps_copy['GracePeriodTakafulRecovery'][-1] - rps_copy['GracePeriodProfitRecovery'][-1],3))
                rps_copy['OutstandingPrincipal'].append(round(rps_copy['OutstandingPrincipal'][-1] - rps_copy['PrincipalAmount'][-1],3))

            if rps_copy['OutstandingPrincipal'][-1] < 0:
                EMI = rps_copy['EMI'][-1] + rps_copy['OutstandingPrincipal'][-1]
                PrincipalAmount = rps_copy['PrincipalAmount'][-1] + rps_copy['OutstandingPrincipal'][-1]
                rps_copy['EMI'][-1] = EMI
                rps_copy['PrincipalAmount'][-1] = PrincipalAmount
                rps_copy['OutstandingPrincipal'][-1] = 0
                break


            if sum(1 for i in rps_copy['PrincipalAmount'] if i < 0)== 0:
                break
            else:
                distributionMonths += 1

        # In case the loan is fully paid off during the distribution period, then we just copy the RPS without adding more rows to it. 
        if rps_copy['OutstandingPrincipal'][-1] > 0:
            
            # Add the rows for the period during which we are recovering the deferred profit and takaful amounts.
            for i in range (0, distributionMonths):
                rps['SNo'].append(2+i) 
                year = FirstEMIYear + ((FirstEMIMonth + (i-1))//12)
                month = ((FirstEMIMonth + (i-1)) % 12) + 1
                day = PayDay
                rps['Date'].append(date(year, month, day))  
                rps['Days'].append((rps['Date'][-1] - rps['Date'][-2]).days)
                rps['ProfitAmount'].append(round(rps['OutstandingPrincipal'][-1] * ProfitRate / 360 * rps['Days'][-1],3))
                # rps['TakafulAmount'].append(rps['OutstandingPrincipal'][-1] * TakafulFactor * rps['Days'][-1] / 30)
                rps['TakafulAmount'].append(round(rps['OutstandingPrincipal'][-1] * TakafulFactor,3))
                rps['GracePeriodTakafulRecovery'].append(round(GracePeriodTakaful / distributionMonths,3))
                rps['GracePeriodProfitRecovery'].append(round(GracePeriodProfit / distributionMonths,3))
                rps['EMI'].append(EMI)
                rps['PrincipalAmount'].append(round(max(0,rps['EMI'][-1] - rps['ProfitAmount'][-1] - rps['TakafulAmount'][-1] - rps['GracePeriodTakafulRecovery'][-1] - rps['GracePeriodProfitRecovery'][-1]),3))
                rps['OutstandingPrincipal'].append(round(rps['OutstandingPrincipal'][-1] - rps['PrincipalAmount'][-1],3))
                    

            i = distributionMonths

            while rps['OutstandingPrincipal'][-1] > 0:
                rps['SNo'].append(rps['SNo'][-1] + 1)
                year = FirstEMIYear + ((FirstEMIMonth + (i-1))//12)
                month = ((FirstEMIMonth + (i-1)) % 12) + 1
                day = PayDay
                rps['Date'].append(date(year, month, day))  
                
                rps['Days'].append((rps['Date'][-1] - rps['Date'][-2]).days)
                
                rps['EMI'].append(EMI)
                rps['ProfitAmount'].append(round(rps['OutstandingPrincipal'][-1] * ProfitRate / 360 * rps['Days'][-1],3))
                # rps['TakafulAmount'].append(rps['OutstandingPrincipal'][-1] * TakafulFactor * rps['Days'][-1] / 30)
                rps['TakafulAmount'].append(round(rps['OutstandingPrincipal'][-1] * TakafulFactor,3))
                rps['GracePeriodTakafulRecovery'].append(0)
                rps['GracePeriodProfitRecovery'].append(0)
                rps['PrincipalAmount'].append(round(max(0,rps['EMI'][-1] - rps['ProfitAmount'][-1] - rps['TakafulAmount'][-1] - rps['GracePeriodTakafulRecovery'][-1] - rps['GracePeriodProfitRecovery'][-1]),3))
                rps['OutstandingPrincipal'].append(round(rps['OutstandingPrincipal'][-1] - rps['PrincipalAmount'][-1],3))

                i += 1

            if rps['OutstandingPrincipal'][-1] < 0:
                EMI = rps['EMI'][-1] + rps['OutstandingPrincipal'][-1]
                PrincipalAmount = rps['PrincipalAmount'][-1] + rps['OutstandingPrincipal'][-1]
                rps['EMI'][-1] = EMI
                rps['PrincipalAmount'][-1] = PrincipalAmount
                rps['OutstandingPrincipal'][-1] = 0
                
        else:
            rps = rps_copy


    else:
        raise AttributeError("Please input the correct repaymentMethod. (Deferral)")
    
    df = pd.DataFrame(rps)

    return df

In [59]:
SeekPrincipal(
        Tenor = 120,
        ProfitRate = 0.05,
        PayDay = 1,
        DisbursementDate = "20260415",
        TakafulFactor  = 0.055,
        EMI = 400,
        GracePeriodMonths = 3,
        repaymentMethod = "Deferral"
        )

ValueError: No valid FinanceAmount exists for EMI=400, Tenor=120, GracePeriodMonths=3. EMI is too low to service the profit and takaful charges.